## Faiss

Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors.It contains algorithms that search in sets of vectors of any size , up to ones that possibly do not fir in RAM.It also contains supporting code for evaluation and parameter tuning.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("speech.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30)
docs = text_splitter.split_documents(documents)


In [ ]:
docs

In [ ]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
db = FAISS.from_documents(docs, embeddings)
db


In [ ]:
#### Querying the database
query = "What does the speaker believe is the main reason the United states should enter the war?"
docs=db.similarity_search(query)
docs[0].page_content

## As a Retriever

In [ ]:
retriever = db.as_retriever()
docs = retriever.invoke(query)
docs[0].page_content

## Similarity Search with score

There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [ ]:
docs_and_scores = db.similarity_search_with_score(query)
docs_and_scores

In [ ]:
embedding_vector = embeddings.embed_query(query)
embedding_vector

In [ ]:
doc_score = db.similarity_search_by_vector(embedding_vector)
doc_score

In [ ]:
###Saving and Local

db.save_local("faiss_index")

In [ ]:
new_db = FAISS.load_local("faiss_index", embeddings,allow_dangerous_deserialization=True)
docs = new_db.similarity_search(query)
docs